In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="text-align:right;padding:8px 0;">
<button id="toggle-code-btn" style="
  padding:8px 18px;font-size:14px;cursor:pointer;
  background:#4a90d9;color:white;border:none;border-radius:5px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2);
" onclick="
  var cells = document.querySelectorAll('.jp-Cell-inputWrapper');
  var btn = document.getElementById('toggle-code-btn');
  var show = btn.dataset.show !== 'true';
  btn.dataset.show = show;
  btn.textContent = show ? '隐藏代码 [Hide]' : '显示代码 [Show]';
  cells.forEach(function(c){ c.style.display = show ? '' : 'none'; });
">显示代码 [Show]</button>
</div>
'''))


# 第12章 软件开发 - 12.1 程序开发生命周期
# Chapter 12 Software Development - 12.1 Program Development Life Cycle

---

**Cambridge International AS & A Level Computer Science (9618)**

在这一节中，我们将深入理解软件开发的完整生命周期——从最初的想法到最终的维护。
这不仅仅是「写代码」那么简单，软件开发是一门系统化的工程。

## 1. 为什么需要开发生命周期？ Why Do We Need a Development Life Cycle?

### 没有计划的灾难

想象一下，你决定建一栋房子：

| 没有蓝图 (Without Blueprints) | 有蓝图 (With Blueprints) |
|---|---|
| 随意开始砌墙，发现门的位置不对 | 先画设计图，确定每面墙的位置 |
| 电线装好后发现水管没地方放 | 提前规划好所有管线的走向 |
| 快完工时发现地基不够稳 | 地基承重在一开始就计算好 |
| 最终：花了两倍的钱，三倍的时间 | 最终：按时按预算完成 |

**软件开发也是一样的！** 如果没有系统的开发流程：
- 需求不明确 → 做出来的东西不是用户想要的
- 没有设计 → 代码杂乱无章，难以维护
- 没有测试 → 上线后到处是 bug
- 没有维护计划 → 软件很快就过时

> **关键洞察 (Key Insight):** 修复一个在需求阶段发现的问题，成本是 1x；
> 在设计阶段发现是 5x；在编码阶段是 10x；在测试阶段是 20x；
> 在上线后发现是 **100x**！这就是为什么我们需要结构化的开发流程。

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

import matplotlib.patches as mpatches
import numpy as np

# 修复 bug 成本随阶段增长的可视化
stages = ['Analysis\n需求分析', 'Design\n设计', 'Coding\n编码', 'Testing\n测试', 'Maintenance\n维护']
costs = [1, 5, 10, 20, 100]
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#8e44ad']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stages, costs, color=colors, edgecolor='black', linewidth=0.5)
for bar, cost in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{cost}x', ha='center', va='bottom', fontsize=14, fontweight='bold')
ax.set_ylabel('Relative Cost (相对成本)', fontsize=12)
ax.set_title('Cost of Fixing Bugs at Different Stages\n在不同阶段修复 Bug 的相对成本', fontsize=14)
ax.set_ylim(0, 120)
plt.tight_layout()
plt.show()


## 2. 开发生命周期的五个阶段 The Five Stages

程序开发生命周期 (Program Development Life Cycle, PDLC) 包含五个核心阶段：

```
Analysis → Design → Coding → Testing → Maintenance
需求分析 → 设计 → 编码 → 测试 → 维护
```

每个阶段都有明确的输入、活动和输出。让我们逐一深入了解。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 3))
ax.set_xlim(0, 15)
ax.set_ylim(0, 3)
ax.axis('off')

stages = [
    ('Analysis\n需求分析', '#2ecc71'),
    ('Design\n设计', '#3498db'),
    ('Coding\n编码', '#f39c12'),
    ('Testing\n测试', '#e74c3c'),
    ('Maintenance\n维护', '#8e44ad')
]

for i, (label, color) in enumerate(stages):
    x = 0.5 + i * 2.9
    rect = mpatches.FancyBboxPatch((x, 0.7), 2.2, 1.6, boxstyle='round,pad=0.1',
                                   facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + 1.1, 1.5, label, ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')
    if i < 4:
        ax.annotate('', xy=(x + 2.5, 1.5), xytext=(x + 2.2, 1.5),
                    arrowprops=dict(arrowstyle='->', lw=2, color='#333'))

ax.set_title('Program Development Life Cycle (PDLC)', fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

### 2.1 需求分析 Analysis

**目标：** 弄清楚「要做什么」（What），而不是「怎么做」（How）。

这是整个生命周期中**最重要**的阶段。如果你不理解问题，怎么可能写出正确的解决方案？

#### 需求分析包含的活动：

| 活动 | 说明 | 类比 |
|---|---|---|
| **需求收集** (Requirements Gathering) | 访谈用户、问卷调查、观察现有流程 | 医生问诊：你哪里不舒服？ |
| **可行性研究** (Feasibility Study) | 技术上能实现吗？经济上划算吗？时间够吗？ | 盖房子前先看地基和预算 |
| **需求文档** (Requirements Specification) | 把所有需求写成正式文档 | 建筑图纸和合同 |

#### 可行性研究的四个维度：
1. **技术可行性** (Technical): 现有技术能实现吗？
2. **经济可行性** (Economic): 投入产出比合理吗？
3. **法律可行性** (Legal): 是否符合法律法规？
4. **时间可行性** (Schedule): 能在截止日期前完成吗？

> **真实案例:** 当 NASA 开发航天飞机软件时，需求分析阶段就花了数年时间——
> 因为一个小错误可能导致宇航员的生命危险。

### 2.2 设计 Design

**目标：** 规划「怎么做」（How），把需求转化为技术方案。

#### 设计阶段的产出：

- **算法设计** (Algorithm Design): 用伪代码 (pseudocode) 或流程图 (flowchart) 描述解决步骤
- **数据结构设计** (Data Structure Design): 选择合适的数据存储方式
- **界面设计** (UI Design): 用户看到什么、怎么交互
- **模块划分** (Modular Design): 把大问题拆成小模块
- **数据库设计** (Database Design): 如果需要数据库，设计表结构

**类比：** 就像建筑师画详细的施工图——不仅要画出外观，还要画出每根钢筋的位置、
每根水管的走向、每条电线的连接方式。

> **重要原则 - 分而治之 (Divide and Conquer):** 把一个复杂的大问题分解成若干个
> 简单的小问题，分别解决后再组合起来。这也叫**分解** (Decomposition)。

### 2.3 编码 Coding

**目标：** 将设计转化为可执行的程序代码。

这是大多数初学者认为「编程」的全部——但实际上它只是整个生命周期的**一小部分**！

#### 编码阶段的关键活动：
- 选择合适的编程语言
- 按照设计文档编写代码
- 编写注释和文档
- 遵循编码规范 (Coding Standards)
- 版本控制 (Version Control) —— 比如使用 Git

> **深层思考:** 好的程序员花在**思考**上的时间远多于**打字**的时间。
> 一行精心设计的代码胜过十行仓促写出的代码。

### 2.4 测试 Testing

**目标：** 发现并修复错误 (Bug)，确保软件符合需求。

测试不是要证明程序「没有错误」——这几乎是不可能的。
测试的目的是**尽可能多地发现错误**。

#### 测试的层次：
1. **单元测试** (Unit Testing): 测试单个模块/函数
2. **集成测试** (Integration Testing): 测试模块间的协作
3. **系统测试** (System Testing): 测试整个系统
4. **验收测试** (Acceptance Testing): 用户确认是否满足需求

> **Dijkstra 名言:** "Testing shows the presence, not the absence of bugs."
> （测试能证明 bug 的存在，但不能证明 bug 的不存在。）

### 2.5 维护 Maintenance

**目标：** 软件上线后持续运行、更新和改进。

**令人惊讶的事实：** 维护阶段通常占软件总成本的 **60-80%**！

维护分为三种类型：

| 类型 | 英文 | 说明 | 例子 |
|---|---|---|---|
| 纠正性维护 | Corrective | 修复上线后发现的 bug | 修复登录偶尔失败的问题 |
| 适应性维护 | Adaptive | 适应新的运行环境 | 适配新版本的操作系统 |
| 完善性维护 | Perfective | 增加功能或提高性能 | 添加用户要求的新报表功能 |

> **类比：** 买了一辆车，交车只是开始——你还需要定期保养（纠正性）、
> 更换适合新路况的轮胎（适应性）、加装倒车雷达（完善性）。

In [ ]:
import matplotlib.pyplot as plt

# 维护成本占比可视化
labels = ['Analysis\n需求分析', 'Design\n设计', 'Coding\n编码', 'Testing\n测试', 'Maintenance\n维护']
sizes = [5, 10, 10, 15, 60]
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#8e44ad']
explode = (0, 0, 0, 0, 0.1)

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(sizes, explode=explode, labels=labels,
    colors=colors, autopct='%1.0f%%', shadow=True, startangle=140,
    textprops={'fontsize': 10})
for t in autotexts:
    t.set_fontsize(11)
    t.set_fontweight('bold')
ax.set_title('Typical Software Cost Distribution\n软件开发各阶段成本占比', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. 开发生命周期模型 Development Life Cycle Models

不同的项目适合不同的开发模型。让我们深入了解三种经典模型。

### 3.1 瀑布模型 Waterfall Model

**核心思想：** 严格的线性顺序，每个阶段必须**完全完成**后才能进入下一阶段。
就像瀑布一样，水只能往下流，不能倒流。

#### 特点：
- 每个阶段有明确的起点和终点
- 每个阶段产出正式的文档
- 只有上一阶段签字确认后，才能开始下一阶段
- 非常结构化、可预测

#### 优点 (Advantages):
- 简单易懂，容易管理
- 文档齐全，每个阶段都有明确的交付物
- 里程碑清晰，进度易于跟踪

#### 缺点 (Disadvantages):
- **不灵活：** 需求一旦确定就很难更改
- **测试太晚：** 到测试阶段才发现设计有问题，代价巨大
- **用户参与少：** 用户在分析阶段之后要等很久才能看到成品
- **风险后移：** 主要风险在项目后期才暴露

#### 最适合：
- 需求非常明确且不会变化的项目
- 安全关键系统（如医疗设备软件、航空系统）
- 合同已经明确定义了所有需求

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 12)
ax.axis('off')

stages = [
    ('Analysis (需求分析)', '#2ecc71'),
    ('Design (设计)', '#3498db'),
    ('Coding (编码)', '#f39c12'),
    ('Testing (测试)', '#e74c3c'),
    ('Maintenance (维护)', '#8e44ad')
]

for i, (label, color) in enumerate(stages):
    x = 1 + i * 1.2
    y = 10 - i * 2
    rect = mpatches.FancyBboxPatch((x, y), 4.5, 1.2, boxstyle='round,pad=0.15',
                                   facecolor=color, edgecolor='black', linewidth=1.5, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x + 2.25, y + 0.6, label, ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')
    if i < 4:
        ax.annotate('', xy=(x + 1.5, y - 0.5), xytext=(x + 1.0, y),
                    arrowprops=dict(arrowstyle='->', lw=2.5, color='#333'))

# 水流效果
for i in range(4):
    x_start = 2.2 + i * 1.2
    y_start = 10 - i * 2
    ax.plot([x_start + 3.5, x_start + 4.0], [y_start + 0.2, y_start - 0.5],
            color='#85c1e9', lw=1.5, alpha=0.5, linestyle='--')

ax.set_title('Waterfall Model (瀑布模型)', fontsize=16, fontweight='bold', pad=15)
ax.text(10, 1, 'Each stage flows\ndown like a waterfall.\nNo going back!\n\n每个阶段像瀑布\n一样向下流动，\n不能回头！',
        fontsize=10, ha='center', va='bottom',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='gray'))
plt.tight_layout()
plt.show()

### 3.2 迭代模型 Iterative Model

**核心思想：** 不是一次做完所有功能，而是**一轮一轮地做**。
每一轮（迭代）都经历分析→设计→编码→测试，每轮产出一个更完善的版本。

**类比：** 就像画一幅油画——
- 第一轮：画出大致轮廓
- 第二轮：填充主要颜色
- 第三轮：添加细节和阴影
- 第四轮：精修和点缀

每一轮结束后，画作都是「完整」的，只是越来越精细。

#### 优点 (Advantages):
- **早期反馈：** 用户很快就能看到可工作的版本
- **灵活适应：** 可以根据反馈调整后续计划
- **风险分散：** 问题在早期就能被发现

#### 缺点 (Disadvantages):
- **范围蔓延 (Scope Creep)：** 用户可能不断要求添加新功能
- **管理复杂：** 需要良好的版本控制和项目管理
- **总体时间可能更长：** 反复修改需要时间

#### 最适合：
- 需求不完全明确或可能变化的项目
- 大型复杂系统
- 需要早期交付部分功能的项目

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.axis('off')
ax.set_aspect('equal')

# 绘制螺旋表示迭代
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
labels_iter = ['Iteration 1\n(第1轮)', 'Iteration 2\n(第2轮)',
               'Iteration 3\n(第3轮)', 'Iteration 4\n(第4轮)']

for i, (c, lab) in enumerate(zip(colors, labels_iter)):
    r = 1.0 + i * 0.9
    theta = np.linspace(0, 2 * np.pi, 100)
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    ax.plot(x, y, color=c, lw=2.5, alpha=0.7)
    # 箭头表示方向
    ax.annotate('', xy=(r * np.cos(0.1), r * np.sin(0.1)),
                xytext=(r * np.cos(-0.1), r * np.sin(-0.1)),
                arrowprops=dict(arrowstyle='->', color=c, lw=2))
    ax.text(r + 0.15, 0.3, lab, fontsize=9, color=c, fontweight='bold')

# 四个阶段标注
phase_labels = ['Analysis', 'Design', 'Coding', 'Testing']
phase_angles = [np.pi/4, 3*np.pi/4, 5*np.pi/4, 7*np.pi/4]
for angle, pl in zip(phase_angles, phase_labels):
    r_text = 4.5
    ax.text(r_text * np.cos(angle), r_text * np.sin(angle), pl,
            ha='center', va='center', fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray'))

ax.set_title('Iterative Model (迭代模型) - Spiral View', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 快速应用开发 Rapid Application Development (RAD)

**核心思想：** 以**最快的速度**交付可工作的软件。
通过快速建立原型 (Prototype)，让用户尽早体验并反馈。

**类比：** 就像做菜——
- 先做一个样品让客人尝尝
- 客人说「太咸了」→ 减盐
- 客人说「加点辣」→ 加辣椒
- 反复调整，直到客人满意

#### RAD 的四个阶段：
1. **需求规划** (Requirements Planning): 快速确定基本需求
2. **用户设计** (User Design): 用户参与设计原型
3. **快速构建** (Rapid Construction): 快速开发原型
4. **切换** (Cutover): 测试、用户培训、部署

#### 优点 (Advantages):
- **快速交付：** 用户很快就能看到成果
- **用户满意度高：** 用户全程参与
- **灵活性强：** 随时根据反馈调整

#### 缺点 (Disadvantages):
- **文档可能不足：** 速度优先，文档往往被忽视
- **不适合大型系统：** 难以扩展到非常复杂的项目
- **依赖用户参与：** 如果用户没时间，项目会受阻
- **质量风险：** 为了速度可能牺牲代码质量

#### 最适合：
- 中小型项目
- 时间紧迫的项目
- 需求可能变化、需要频繁用户反馈的项目

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

# RAD 循环
rad_steps = [
    (1, 5.5, 'Requirements\nPlanning\n需求规划', '#2ecc71'),
    (5, 5.5, 'User Design\n用户设计', '#3498db'),
    (9, 5.5, 'Rapid\nConstruction\n快速构建', '#f39c12'),
    (9, 1.5, 'Cutover\n切换部署', '#e74c3c'),
]

for x, y, label, color in rad_steps:
    rect = mpatches.FancyBboxPatch((x, y), 3.5, 2, boxstyle='round,pad=0.15',
                                   facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + 1.75, y + 1, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

# 箭头连接
ax.annotate('', xy=(5, 6.5), xytext=(4.5, 6.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#333'))
ax.annotate('', xy=(9, 6.5), xytext=(8.5, 6.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#333'))
ax.annotate('', xy=(10.75, 5.5), xytext=(10.75, 3.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#333'))

# 反馈循环
ax.annotate('', xy=(8.5, 7.0), xytext=(5.5, 7.0),
            arrowprops=dict(arrowstyle='->', lw=2, color='#e67e22',
                           connectionstyle='arc3,rad=-0.3', linestyle='dashed'))
ax.text(7, 7.7, 'Feedback Loop (反馈循环)', ha='center', fontsize=10,
        color='#e67e22', fontstyle='italic')

# Prototype 标注
ax.text(1, 2, 'Prototype\n(原型)', fontsize=12, ha='left',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor='orange', lw=1.5))
ax.annotate('', xy=(5, 5.5), xytext=(3.5, 2.8),
            arrowprops=dict(arrowstyle='->', lw=1.5, color='orange', linestyle='dashed'))

ax.set_title('Rapid Application Development (RAD) - 快速应用开发', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. 三种模型对比 Comparison of Three Models

| 特性 | Waterfall 瀑布 | Iterative 迭代 | RAD 快速开发 |
|---|---|---|---|
| **流程** | 线性、顺序 | 循环、螺旋 | 原型驱动 |
| **灵活性** | 低 | 中 | 高 |
| **用户参与** | 仅在分析阶段 | 每次迭代 | 全程参与 |
| **文档** | 非常详细 | 适中 | 较少 |
| **风险发现** | 晚期 | 早期 | 早期 |
| **需求变化** | 难以适应 | 可以适应 | 容易适应 |
| **交付速度** | 慢（一次性交付） | 渐进式交付 | 快速交付 |
| **适合项目** | 大型、需求明确 | 大型、需求演进 | 中小型、时间紧 |
| **典型应用** | 军事、航天系统 | 企业级软件 | Web 应用、APP |

## 5. 互动练习：选择合适的开发模型 Interactive: Choose the Right Model

下面是一个交互式工具，输入项目的特征，它会推荐最合适的开发模型。

In [ ]:
def recommend_model():
    print('='*60)
    print('  Development Model Selector (开发模型选择器)')
    print('='*60)
    print()
    
    score_waterfall = 0
    score_iterative = 0
    score_rad = 0
    
    # Question 1
    print('Q1: 需求是否明确且不太可能变化？')
    print('    Are requirements well-defined and unlikely to change?')
    print('    1 = Yes  /  2 = Partially  /  3 = No')
    q1 = 1  # 模拟回答
    print(f'    Your answer: {q1}')
    if q1 == 1: score_waterfall += 3
    elif q1 == 2: score_iterative += 2
    else: score_rad += 3
    
    # Question 2
    print('\nQ2: 项目规模有多大？')
    print('    How large is the project?')
    print('    1 = Large  /  2 = Medium  /  3 = Small')
    q2 = 2
    print(f'    Your answer: {q2}')
    if q2 == 1: score_waterfall += 2; score_iterative += 2
    elif q2 == 2: score_iterative += 2; score_rad += 1
    else: score_rad += 3
    
    # Question 3
    print('\nQ3: 时间压力有多大？')
    print('    How tight is the deadline?')
    print('    1 = Relaxed  /  2 = Normal  /  3 = Very tight')
    q3 = 3
    print(f'    Your answer: {q3}')
    if q3 == 1: score_waterfall += 2
    elif q3 == 2: score_iterative += 2
    else: score_rad += 3
    
    # Question 4
    print('\nQ4: 用户是否能积极参与开发过程？')
    print('    Can users actively participate in development?')
    print('    1 = No  /  2 = Sometimes  /  3 = Yes, always')
    q4 = 3
    print(f'    Your answer: {q4}')
    if q4 == 1: score_waterfall += 2
    elif q4 == 2: score_iterative += 2
    else: score_rad += 3
    
    print('\n' + '='*60)
    print('  Results (结果):')
    print(f'    Waterfall Score: {score_waterfall}')
    print(f'    Iterative Score: {score_iterative}')
    print(f'    RAD Score:       {score_rad}')
    print()
    
    winner = max([(score_waterfall, 'Waterfall (瀑布模型)'),
                  (score_iterative, 'Iterative (迭代模型)'),
                  (score_rad, 'RAD (快速应用开发)')], key=lambda x: x[0])
    print(f'  Recommended Model: {winner[1]}')
    print('='*60)

recommend_model()

## 6. 练习题 Practice Exercises

### 练习 1: 场景分析 Scenario Analysis

阅读以下项目描述，选择最合适的开发模型并解释原因。

**场景 A:** 一家银行要开发一个核心交易系统。所有的需求已经由监管机构严格定义，
必须满足 100% 的合规要求。系统必须有详尽的文档。

**场景 B:** 一个创业公司要开发一款社交 APP。他们对用户需求只有大致的概念，
计划先推出最小可行产品 (MVP)，然后根据用户反馈不断改进。

**场景 C:** 一个学校需要一个简单的图书管理系统。预算有限，校长希望
在两个月内看到成果，并且愿意每周参加进度会议。

---

### 练习 2: 阶段识别 Stage Identification

以下每个活动属于开发生命周期的哪个阶段？

1. 程序员正在修复一个用户报告的 bug → ?
2. 团队正在画系统的流程图 → ?
3. 项目经理正在和客户讨论系统需要哪些功能 → ?
4. 测试人员正在检查每个按钮是否能正常点击 → ?
5. 开发团队正在把设计好的算法转化为 Python 代码 → ?

In [ ]:
# 练习 1 参考答案
print('练习 1 参考答案：')
print('-' * 50)
print('场景 A: Waterfall (瀑布模型)')
print('  原因: 需求已经被严格定义且不会变化；')
print('  银行系统需要详尽文档；监管合规要求高。')
print()
print('场景 B: Iterative (迭代模型)')
print('  原因: 需求不明确，需要逐步探索；')
print('  计划通过多次迭代逐渐完善产品；')
print('  MVP 策略本身就是迭代思想的体现。')
print()
print('场景 C: RAD (快速应用开发)')
print('  原因: 项目较小；时间紧迫(两个月)；')
print('  用户(校长)愿意全程参与反馈。')
print()
print('='*50)
print('练习 2 参考答案：')
print('-' * 50)
print('1. Maintenance (维护) - 纠正性维护')
print('2. Design (设计)')
print('3. Analysis (需求分析)')
print('4. Testing (测试)')
print('5. Coding (编码)')

### 练习 3: 思考题 Thinking Questions

1. 为什么说「维护阶段是成本最高的阶段」？试从人力、时间和技术三个角度分析。

2. 如果一个项目在开发过程中需求发生了重大变化，使用瀑布模型会遇到什么问题？
   迭代模型如何解决这个问题？

3. 有人说「RAD 模型不适合安全关键系统」，你同意吗？为什么？

4. 现代软件开发中，很多公司使用 Agile（敏捷开发）——它和迭代模型有什么联系和区别？
   （提示：可以课后自行搜索了解 Scrum、Kanban 等方法。）

---

> **本节小结 (Summary):**
> - 开发生命周期包含五个阶段：分析→设计→编码→测试→维护
> - 三种经典模型各有优缺点，要根据项目特点选择
> - **Waterfall** = 线性顺序，适合需求稳定的项目
> - **Iterative** = 循环迭代，适合需求演进的大型项目
> - **RAD** = 快速原型，适合时间紧、用户参与度高的中小项目
> - 维护是成本最高的阶段，好的设计能降低维护成本